Sampling points 

In [56]:
# ---------------------------------------------------------------------
# IMPORTS + EARTH ENGINE AUTHENTICATION
# ---------------------------------------------------------------------
import ee
import math
import pandas as pd
import numpy as np

ee.Authenticate(
    auth_mode="localhost",
    force=True,
)

ee.Initialize(
    project="bop-nca-data-space"
)

print("Earth Engine initialized.")


Successfully saved authorization token.
Earth Engine initialized.


In [57]:
# ---------------------------------------------------------------------
# SETTINGS — MATCH 2018 HARMONIC SENSITIVITY WORKFLOW
# ---------------------------------------------------------------------

YEARS = [2016, 2025]
ORDERS = [2, 3]

TARGET_SCALE = 10
TARGET_CRS = "EPSG:26911"

USE_SCL_MASK = True
USE_CLOUD_PROB_MASK = True

CLOUD_PROB_THRESH = 40
SCENE_CLOUD_PCT = 95

APPLY_MIN_NOBS_MASK = True
MIN_NOBS = 12

USE_CAP_FLOOR_MASK = True

# Original production settings
CAP_FLOOR_SAMPLE_PER_IMAGE = 250
CAP_FLOOR_SAMPLE_SCALE = 20

HISTORICAL_TIME_TOLERANCE_MILLIS = (
    5 * 60 * 1000
)


# ---------------------------------------------------------------------
# COLLECTIONS
# ---------------------------------------------------------------------

HISTORICAL_ASSET = (
    "projects/bop-nca-data-space/"
    "assets/S2_C1_L2A_2016_2017"
)

S2_NATIVE = (
    "COPERNICUS/S2_SR_HARMONIZED"
)

S2_CLOUD_PROB = (
    "COPERNICUS/S2_CLOUD_PROBABILITY"
)


# ---------------------------------------------------------------------
# MODEL VARIABLES — IDENTICAL TO 2018
# ---------------------------------------------------------------------

RAW_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12",
]

Y_BANDS = RAW_BANDS + [
    "NDVI",
    "NDMI",
    "NDRE",
    "BSI",
    "NBR2",
]

In [58]:
POINT_CSV = Path(
    r"N:\Data02\projects-active\BOPclassification_2025"
    r"\Field Data (Processed)\2026"
    r"\MasterPointFeatures_2016to2026.csv"
)


points = pd.read_csv(
    POINT_CSV
)


def normalize_plot_id(series):

    return (
        series.astype(str)
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True,
        )
    )


points["PlotID"] = normalize_plot_id(
    points["PlotID"]
)


points = (
    points.loc[
        points["Year"].isin(YEARS),
        [
            "PlotID",
            "Year",
            "X",
            "Y",
        ],
    ]
    .dropna()
    .drop_duplicates(
        subset=[
            "PlotID",
            "Year",
        ]
    )
    .reset_index(drop=True)
)


display(
    points.groupby("Year")
    .size()
    .rename("n_plots")
)

Year
2016    378
2025    118
Name: n_plots, dtype: int64

In [59]:
def dataframe_to_ee_points(df):

    features = []

    proj = ee.Projection(
        TARGET_CRS
    )

    for row in df.itertuples():

        geom = ee.Geometry.Point(
            [
                float(row.X),
                float(row.Y),
            ],
            proj=proj,
        )

        features.append(
            ee.Feature(
                geom,
                {
                    "PlotID": str(row.PlotID),
                    "Year": int(row.Year),
                },
            )
        )

    return ee.FeatureCollection(
        features
    )


point_fc = dataframe_to_ee_points(
    points
)


print(
    "Total plots:",
    point_fc.size().getInfo()
)

Total plots: 496


In [60]:
# ---------------------------------------------------------------------
# FIXED 2016 COLLECTION BUILD
#
# Join custom 2016 L2A to S2 cloud probability using:
#   1. exact/near acquisition time
#   2. MGRS tile parsed from cloud-probability system:index
#
# Unmatched historical images are retained.
# ---------------------------------------------------------------------

def build_2016_collection(
    year_points,
):

    start = ee.Date(
        "2016-01-01"
    )

    end = ee.Date(
        "2017-01-01"
    )

    point_geom = (
        year_points.geometry()
    )


    # ------------------------------------------------------------
    # CUSTOM 2016 L2A
    # ------------------------------------------------------------

    historical_sr = (
        ee.ImageCollection(
            HISTORICAL_ASSET
        )
        .filterBounds(
            point_geom
        )
        .filterDate(
            start,
            end,
        )
    )


    # ------------------------------------------------------------
    # CLOUD PROBABILITY
    #
    # Parse tile from system:index, e.g.
    #
    # 20160103T185122_20160103T185116_T11TNH
    #
    # -> 11TNH
    # ------------------------------------------------------------

    def add_cp_tile(img):

        img = ee.Image(
            img
        )

        index = ee.String(
            img.get(
                "system:index"
            )
        )

        tile = (
            index
            .split("_")
            .get(-1)
        )

        tile = (
            ee.String(tile)
            .replace(
                "^T",
                ""
            )
        )

        return img.set(
            "MGRS_TILE_JOIN",
            tile,
        )


    s2clouds = (
        ee.ImageCollection(
            S2_CLOUD_PROB
        )
        .filterBounds(
            point_geom
        )
        .filterDate(
            start,
            end,
        )
        .map(
            add_cp_tile
        )
    )


    # ------------------------------------------------------------
    # JOIN
    #
    # Historical:
    #   MGRS_TILE = 11TNH
    #
    # Cloud probability:
    #   MGRS_TILE_JOIN = 11TNH
    #
    # Plus acquisition time within 5 minutes.
    # ------------------------------------------------------------

    join_filter = ee.Filter.And(

        ee.Filter.equals(
            leftField="MGRS_TILE",
            rightField="MGRS_TILE_JOIN",
        ),

        ee.Filter.maxDifference(
            difference=(
                HISTORICAL_TIME_TOLERANCE_MILLIS
            ),
            leftField="system:time_start",
            rightField="system:time_start",
        ),
    )


    joined = ee.ImageCollection(
        ee.Join.saveFirst(
            matchKey="cloudprob",
            outer=True,
        ).apply(
            primary=historical_sr,
            secondary=s2clouds,
            condition=join_filter,
        )
    )


    # ------------------------------------------------------------
    # ATTACH CLOUD PROBABILITY WHEN AVAILABLE
    # ------------------------------------------------------------

    def attach_cloud_probability(img):

        img = ee.Image(
            img
        )

        cloud_obj = (
            img.get(
                "cloudprob"
            )
        )

        has_match = ee.Number(
            ee.Algorithms.If(
                ee.Algorithms.IsEqual(
                    cloud_obj,
                    None,
                ),
                0,
                1,
            )
        )


        cloud_prob = ee.Image(
            ee.Algorithms.If(

                has_match.eq(1),

                ee.Image(
                    cloud_obj
                )
                .select(
                    "probability"
                )
                .rename(
                    "cloudProb"
                ),

                ee.Image.constant(
                    0
                )
                .rename(
                    "cloudProb"
                ),
            )
        )


        return (
            img
            .addBands(
                cloud_prob
            )
            .set(
                "cp_matched",
                has_match,
            )
            .set(
                "source_group",
                "historical_cdse",
            )
        )


    return joined.map(
        attach_cloud_probability
    )

In [61]:
year_points_2016 = (
    point_fc
    .filter(
        ee.Filter.eq(
            "Year",
            2016,
        )
    )
)

test_2016 = build_2016_collection(
    year_points_2016
)

n_total = (
    test_2016
    .size()
)

n_matched = (
    test_2016
    .filter(
        ee.Filter.eq(
            "cp_matched",
            1,
        )
    )
    .size()
)

n_unmatched = (
    test_2016
    .filter(
        ee.Filter.eq(
            "cp_matched",
            0,
        )
    )
    .size()
)

print(
    "2016 source images:",
    n_total.getInfo()
)

print(
    "CP matched:",
    n_matched.getInfo()
)

print(
    "CP unmatched:",
    n_unmatched.getInfo()
)

print(
    "Matched + unmatched == total:",
    n_matched
    .add(n_unmatched)
    .eq(n_total)
    .getInfo()
)

2016 source images: 202
CP matched: 187
CP unmatched: 15
Matched + unmatched == total: 1


In [62]:
# ---------------------------------------------------------------------
# DIAGNOSE 2016 CLOUD-PROBABILITY JOIN KEYS
# ---------------------------------------------------------------------

year_points_2016 = (
    point_fc
    .filter(
        ee.Filter.eq(
            "Year",
            2016,
        )
    )
)

point_geom_2016 = (
    year_points_2016
    .geometry()
)


# ------------------------------------------------------------
# RAW CUSTOM 2016 COLLECTION
# ------------------------------------------------------------

hist = (
    ee.ImageCollection(
        HISTORICAL_ASSET
    )
    .filterBounds(
        point_geom_2016
    )
    .filterDate(
        "2016-01-01",
        "2017-01-01",
    )
)


# ------------------------------------------------------------
# CLOUD PROBABILITY COLLECTION
# ------------------------------------------------------------

cp = (
    ee.ImageCollection(
        S2_CLOUD_PROB
    )
    .filterBounds(
        point_geom_2016
    )
    .filterDate(
        "2016-01-01",
        "2017-01-01",
    )
)


print(
    "Historical images:",
    hist.size().getInfo()
)

print(
    "Cloud-probability images:",
    cp.size().getInfo()
)


# ------------------------------------------------------------
# FIRST CUSTOM IMAGE
# ------------------------------------------------------------

hist_first = ee.Image(
    hist.first()
)

print(
    "\nCUSTOM IMAGE PROPERTIES:"
)

print(
    hist_first
    .propertyNames()
    .getInfo()
)

print(
    "\nCustom system:index:",
    hist_first.get(
        "system:index"
    ).getInfo()
)

print(
    "Custom MGRS_TILE:",
    hist_first.get(
        "MGRS_TILE"
    ).getInfo()
)

print(
    "Custom PRODUCT_ID:",
    hist_first.get(
        "PRODUCT_ID"
    ).getInfo()
)

print(
    "Custom time:",
    ee.Date(
        hist_first.get(
            "system:time_start"
        )
    ).format(
        "YYYY-MM-dd HH:mm:ss"
    ).getInfo()
)


# ------------------------------------------------------------
# FIND CLOUD-PROBABILITY IMAGES NEAR SAME TIME
# ------------------------------------------------------------

hist_time = ee.Date(
    hist_first.get(
        "system:time_start"
    )
)

nearby_cp = (
    cp
    .filterDate(
        hist_time.advance(
            -10,
            "minute",
        ),
        hist_time.advance(
            10,
            "minute",
        ),
    )
)


print(
    "\nCloud-probability images within +/- 10 min:",
    nearby_cp.size().getInfo()
)


cp_list = (
    nearby_cp
    .toList(
        nearby_cp.size()
    )
)


n_nearby = (
    nearby_cp
    .size()
    .getInfo()
)


for i in range(
    min(
        n_nearby,
        10,
    )
):

    img = ee.Image(
        cp_list.get(i)
    )

    print(
        "\nCP IMAGE",
        i
    )

    print(
        "system:index:",
        img.get(
            "system:index"
        ).getInfo()
    )

    print(
        "MGRS_TILE:",
        img.get(
            "MGRS_TILE"
        ).getInfo()
    )

    print(
        "PRODUCT_ID:",
        img.get(
            "PRODUCT_ID"
        ).getInfo()
    )

    print(
        "time:",
        ee.Date(
            img.get(
                "system:time_start"
            )
        ).format(
            "YYYY-MM-dd HH:mm:ss"
        ).getInfo()
    )

Historical images: 202
Cloud-probability images: 373

CUSTOM IMAGE PROPERTIES:
['SATELLITE', 'system:time_end', 'system:id', 'source', 'PRODUCT_ID', 'QA60_RECONSTRUCTED', 'system:time_start', 'source_group', 'PROCESSING_BASELINE', 'system:footprint', 'system:version', 'system:asset_size', 'MGRS_TILE', 'system:index', 'system:bands', 'system:band_names']

Custom system:index: S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_20231014T014631
Custom MGRS_TILE: 11TNH
Custom PRODUCT_ID: S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_20231014T014631
Custom time: 2016-01-03 18:51:22

Cloud-probability images within +/- 10 min: 6

CP IMAGE 0
system:index: 20160103T185122_20160103T185116_T11TNH
MGRS_TILE: None
PRODUCT_ID: None
time: 2016-01-03 18:51:22

CP IMAGE 1
system:index: 20160103T185122_20160103T185116_T11TNJ
MGRS_TILE: None
PRODUCT_ID: None
time: 2016-01-03 18:51:22

CP IMAGE 2
system:index: 20160103T185122_20160103T185116_T11TPH
MGRS_TILE: None
PRODUCT_ID: None
time: 2016-01-03 18:51:22

CP I

KeyboardInterrupt: 

In [64]:
def build_2025_collection(
    year_points,
):

    start = ee.Date(
        "2025-01-01"
    )

    end = ee.Date(
        "2026-01-01"
    )

    point_geom = (
        year_points.geometry()
    )


    s2sr = (
        ee.ImageCollection(
            S2_NATIVE
        )
        .filterBounds(
            point_geom
        )
        .filterDate(
            start,
            end,
        )
        .filter(
            ee.Filter.lte(
                "CLOUDY_PIXEL_PERCENTAGE",
                SCENE_CLOUD_PCT,
            )
        )
    )


    s2clouds = (
        ee.ImageCollection(
            S2_CLOUD_PROB
        )
        .filterBounds(
            point_geom
        )
        .filterDate(
            start,
            end,
        )
    )


    join_filter = (
        ee.Filter.equals(
            leftField="system:index",
            rightField="system:index",
        )
    )


    joined = ee.ImageCollection(
        ee.Join.saveFirst(
            "cloudprob"
        ).apply(
            primary=s2sr,
            secondary=s2clouds,
            condition=join_filter,
        )
    )


    # Keep same production behavior for native collection:
    # require the cloud-probability match.

    joined = joined.filter(
        ee.Filter.notNull([
            "cloudprob"
        ])
    )


    def attach_cloud_probability(img):

        img = ee.Image(img)

        cloud_prob = (
            ee.Image(
                img.get(
                    "cloudprob"
                )
            )
            .select(
                "probability"
            )
            .rename(
                "cloudProb"
            )
        )

        return (
            img
            .addBands(
                cloud_prob
            )
            .set(
                "cp_matched",
                1,
            )
            .set(
                "source_group",
                "native_gee",
            )
        )


    return joined.map(
        attach_cloud_probability
    )

In [65]:
def get_year_collection(
    year,
    year_points,
):

    if year == 2016:

        return build_2016_collection(
            year_points
        )

    elif year == 2025:

        return build_2025_collection(
            year_points
        )

    else:

        raise ValueError(
            f"No collection configured "
            f"for {year}"
        )

In [66]:
def scale_sr(img):

    img = ee.Image(
        img
    )

    optical = (
        img
        .select(
            RAW_BANDS
        )
        .multiply(
            0.0001
        )
    )

    return img.addBands(
        optical,
        None,
        True,
    )

In [67]:
def mask_clouds_shadows(img):

    img = ee.Image(
        img
    )


    # ------------------------------------------------------------
    # QA60
    # ------------------------------------------------------------

    qa = img.select(
        "QA60"
    )

    cloud_bit = (
        1 << 10
    )

    cirrus_bit = (
        1 << 11
    )

    qa_mask = (
        qa
        .bitwiseAnd(
            cloud_bit
        )
        .eq(0)
        .And(
            qa
            .bitwiseAnd(
                cirrus_bit
            )
            .eq(0)
        )
    )


    mask = qa_mask


    # ------------------------------------------------------------
    # SCL
    # ------------------------------------------------------------

    if USE_SCL_MASK:

        scl = img.select(
            "SCL"
        )

        scl_mask = (
            scl
            .neq(0)
            .And(scl.neq(1))
            .And(scl.neq(3))
            .And(scl.neq(7))
            .And(scl.neq(8))
            .And(scl.neq(9))
            .And(scl.neq(10))
            .And(scl.neq(11))
        )

        mask = mask.And(
            scl_mask
        )


    # ------------------------------------------------------------
    # CLOUD PROBABILITY
    #
    # Apply only where a match exists.
    #
    # 2016 unmatched image:
    #   cp_matched = 0
    #   -> cloud-probability test becomes TRUE
    #   -> QA60 + SCL still operate normally.
    # ------------------------------------------------------------

    if USE_CLOUD_PROB_MASK:

        cp_matched = ee.Number(
            img.get(
                "cp_matched"
            )
        )

        cp_mask = ee.Image(
            ee.Algorithms.If(

                cp_matched.eq(1),

                img
                .select(
                    "cloudProb"
                )
                .lt(
                    CLOUD_PROB_THRESH
                ),

                ee.Image.constant(
                    1
                ),
            )
        )

        mask = mask.And(
            cp_mask
        )


    return img.updateMask(
        mask
    )

In [68]:
def select_bands(img):

    return (
        ee.Image(img)
        .select(
            RAW_BANDS
        )
    )

In [69]:
def safe_div(
    num,
    den,
):

    valid = (
        den
        .abs()
        .gt(
            1e-6
        )
    )

    return (
        num
        .divide(den)
        .updateMask(
            valid
        )
    )


def add_indices(img):

    img = ee.Image(
        img
    )

    B2 = img.select("B2")
    B4 = img.select("B4")
    B6 = img.select("B6")
    B8 = img.select("B8")

    B11 = img.select("B11")
    B12 = img.select("B12")


    NDVI = safe_div(
        B8.subtract(B4),
        B8.add(B4),
    ).rename(
        "NDVI"
    )


    NDMI = safe_div(
        B8.subtract(B11),
        B8.add(B11),
    ).rename(
        "NDMI"
    )


    NDRE = safe_div(
        B8.subtract(B6),
        B8.add(B6),
    ).rename(
        "NDRE"
    )


    BSI = safe_div(

        B11
        .add(B4)
        .subtract(
            B8.add(B2)
        ),

        B11
        .add(B4)
        .add(B8)
        .add(B2),

    ).rename(
        "BSI"
    )


    NBR2 = safe_div(
        B11.subtract(B12),
        B11.add(B12),
    ).rename(
        "NBR2"
    )


    return img.addBands([
        NDVI,
        NDMI,
        NDRE,
        BSI,
        NBR2,
    ])

In [70]:
def dedupe_by_date(ic):

    date_list = (
        ee.List(
            ic.aggregate_array(
                "system:time_start"
            )
        )
        .map(
            lambda t:
            ee.Date(t)
            .format(
                "YYYY-MM-dd"
            )
        )
        .distinct()
        .sort()
    )


    def make_daily(d):

        d = ee.String(
            d
        )

        day_start = (
            ee.Date.parse(
                "YYYY-MM-dd",
                d,
            )
        )

        day_end = (
            day_start.advance(
                1,
                "day",
            )
        )

        day_col = ic.filterDate(
            day_start,
            day_end,
        )

        return (
            day_col
            .median()
            .set(
                "system:time_start",
                day_start.millis(),
            )
            .set(
                "date",
                d,
            )
            .set(
                "n_daily_images",
                day_col.size(),
            )
        )


    return ee.ImageCollection.fromImages(
        date_list.map(
            make_daily
        )
    )

In [71]:
# ---------------------------------------------------------------------
# AOI
#
# Used only for reproducing the annual cap/floor threshold calculation
# from the full-raster harmonic workflow.
#
# Replace the asset ID below with the exact NCA AOI asset used in the
# production harmonic script.
# ---------------------------------------------------------------------

AOI_ASSET = (
    "projects/bop-nca-data-space/assets/NCA_5kmBuffer_NAD83TM"
)

aoi_fc = ee.FeatureCollection(
    AOI_ASSET
)

aoi = aoi_fc.geometry()


print(
    "AOI features:",
    aoi_fc.size().getInfo()
)

print(
    "AOI area (ha):",
    aoi.area().divide(10000).getInfo()
)

AOI features: 1
AOI area (ha): 444895.7339939789


In [72]:
def compute_annual_cap_floor_thresholds(
    year,
    year_points,
):

    start = ee.Date.fromYMD(
        year,
        1,
        1,
    )

    end = start.advance(
        1,
        "year",
    )


    annual_base = (
        get_year_collection(
            year,
            year_points,
        )
        .filterDate(
            start,
            end,
        )
        .map(
            scale_sr
        )
        .map(
            mask_clouds_shadows
        )
        .map(
            select_bands
        )
        .select([
            "B2",
            "B11",
        ])
    )


    annual_daily = (
        dedupe_by_date(
            annual_base
        )
    )


    def sample_image(img):

        img = ee.Image(
            img
        )

        seed = (
            ee.Number(
                img.get(
                    "system:time_start"
                )
            )
            .mod(
                2147483647
            )
        )

        return img.sample(
            region=aoi,
            scale=(
                CAP_FLOOR_SAMPLE_SCALE
            ),
            numPixels=(
                CAP_FLOOR_SAMPLE_PER_IMAGE
            ),
            geometries=False,
            seed=seed,
            tileScale=4,
        )


    annual_samples = (
        annual_daily
        .map(
            sample_image
        )
        .flatten()
    )


    b2_pct = (
        annual_samples
        .reduceColumns(
            reducer=(
                ee.Reducer.percentile([
                    98
                ])
            ),
            selectors=[
                "B2"
            ],
        )
    )


    b11_pct = (
        annual_samples
        .reduceColumns(
            reducer=(
                ee.Reducer.percentile([
                    2
                ])
            ),
            selectors=[
                "B11"
            ],
        )
    )


    return ee.Dictionary({
        "blueCap":
            b2_pct.get(
                "p98"
            ),

        "swirFloor":
            b11_pct.get(
                "p2"
            ),
    })

In [73]:
def make_annual_cap_floor_mask(
    thresholds,
):

    blue_cap = ee.Number(
        thresholds.get(
            "blueCap"
        )
    )

    swir_floor = ee.Number(
        thresholds.get(
            "swirFloor"
        )
    )


    def apply_mask(img):

        if not USE_CAP_FLOOR_MASK:
            return img

        img = ee.Image(
            img
        )

        blue_ok = (
            img
            .select("B2")
            .lt(
                blue_cap
            )
        )

        swir_ok = (
            img
            .select("B11")
            .gt(
                swir_floor
            )
        )

        return img.updateMask(
            blue_ok.And(
                swir_ok
            )
        )


    return apply_mask

In [74]:
def make_x_bands(
    order,
):

    xb = [
        "constant"
    ]

    for k in range(
        1,
        order + 1,
    ):

        xb.extend([
            f"sin{k}",
            f"cos{k}",
        ])

    return xb


def add_time_terms_for_year(
    img,
    year,
    order,
):

    img = ee.Image(
        img
    )

    date = ee.Date(
        img.get(
            "system:time_start"
        )
    )

    start = ee.Date.fromYMD(
        year,
        1,
        1,
    )

    t = (
        date
        .difference(
            start,
            "day",
        )
        .divide(
            365.25
        )
    )

    t_img = (
        ee.Image.constant(
            t
        )
        .float()
    )

    out = img.addBands(
        ee.Image.constant(
            1
        )
        .rename(
            "constant"
        )
    )


    for k in range(
        1,
        order + 1,
    ):

        angle = t_img.multiply(
            2.0
            * math.pi
            * k
        )

        out = out.addBands([
            angle
            .sin()
            .rename(
                f"sin{k}"
            ),

            angle
            .cos()
            .rename(
                f"cos{k}"
            ),
        ])


    return out

In [75]:
# ---------------------------------------------------------------------
# CELL 15 — PREPARE YEAR-SPECIFIC COLLECTIONS
#
# Point-only harmonic sensitivity workflow.
# Annual B2 cap / B11 floor masks are intentionally skipped here.
# ---------------------------------------------------------------------

prepared_years = {}


for year in YEARS:

    year_points = (
        point_fc
        .filter(
            ee.Filter.eq(
                "Year",
                year,
            )
        )
    )


    print(
        "\n" + "=" * 60
    )

    print(
        "YEAR",
        year
    )


    source = get_year_collection(
        year,
        year_points,
    )


    print(
        "Source images:",
        source.size().getInfo()
    )


    # ------------------------------------------------------------
    # CLOUD-PROBABILITY JOIN QA
    # ------------------------------------------------------------

    matched = (
        source
        .filter(
            ee.Filter.eq(
                "cp_matched",
                1,
            )
        )
        .size()
    )

    unmatched = (
        source
        .filter(
            ee.Filter.eq(
                "cp_matched",
                0,
            )
        )
        .size()
    )


    print(
        "Cloud-probability matched:",
        matched.getInfo()
    )

    print(
        "Cloud-probability unmatched:",
        unmatched.getInfo()
    )


    # ------------------------------------------------------------
    # PREPROCESSING
    #
    # Same core workflow as the raster harmonic script, except:
    # annual cap/floor filtering is omitted for this point-only
    # K2/K3 sensitivity experiment.
    # ------------------------------------------------------------

    base = (
        source
        .map(
            scale_sr
        )
        .map(
            mask_clouds_shadows
        )
        .map(
            select_bands
        )
    )


    daily = (
        dedupe_by_date(
            base
        )
    )


    processed = (
        daily
        .map(
            add_indices
        )
    )


    prepared_years[year] = {
        "points":
            year_points,

        "collection":
            processed,
    }


    print(
        "Daily observations:",
        processed.size().getInfo()
    )


YEAR 2016
Source images: 202
Cloud-probability matched: 187
Cloud-probability unmatched: 15
Daily observations: 76

YEAR 2025
Source images: 376
Cloud-probability matched: 376
Cloud-probability unmatched: 0
Daily observations: 149


In [76]:
# ---------------------------------------------------------------------
# CELL 16
# FIT INDEPENDENT K2 + K3 OLS HARMONIC MODELS
# AND SAMPLE ONLY THE YEAR-MATCHED FIELD POINTS
# ---------------------------------------------------------------------

sampled_tables = {}


for year in YEARS:

    year_points = prepared_years[year]["points"]
    processed = prepared_years[year]["collection"]

    print(
        "\n" + "=" * 60
    )

    print(
        f"YEAR {year}"
    )

    print(
        "Plots:",
        year_points.size().getInfo()
    )

    print(
        "Processed daily observations:",
        processed.size().getInfo()
    )


    for order in ORDERS:

        print(
            f"\nFitting K{order} ..."
        )


        # ------------------------------------------------------------
        # ORDER-SPECIFIC DESIGN MATRIX
        #
        # K2:
        #   constant, sin1, cos1, sin2, cos2
        #
        # K3:
        #   constant, sin1, cos1, sin2, cos2, sin3, cos3
        #
        # IMPORTANT:
        # K2 and K3 are fit independently from the same processed
        # Sentinel-2 observations. K3 is NOT K2 with frozen lower-
        # order coefficients plus two additional coefficients.
        # ------------------------------------------------------------

        x_bands = make_x_bands(
            order
        )


        # ------------------------------------------------------------
        # ADD HARMONIC TIME TERMS
        # ------------------------------------------------------------

        with_x = (
            processed
            .map(
                lambda img:
                add_time_terms_for_year(
                    img,
                    year,
                    order,
                )
            )
        )


        # ------------------------------------------------------------
        # NUMBER OF VALID OBSERVATIONS PER PIXEL
        # ------------------------------------------------------------

        nobs = (
            with_x
            .select("B2")
            .count()
            .round()
            .toInt16()
            .rename("nobs")
        )


        fit_mask = (
            nobs.gte(
                MIN_NOBS
            )
        )


        # Apply the same minimum-observation rule used in the
        # harmonic sensitivity workflow.

        if APPLY_MIN_NOBS_MASK:

            with_x = (
                with_x
                .map(
                    lambda img:
                    ee.Image(img)
                    .updateMask(
                        fit_mask
                    )
                )
            )


        # ------------------------------------------------------------
        # MULTIVARIATE OLS
        #
        # X = harmonic predictors
        # Y = 10 raw bands + 5 indices
        # ------------------------------------------------------------

        regression = (
            with_x
            .select(
                x_bands
                + Y_BANDS
            )
            .reduce(
                ee.Reducer.linearRegression(
                    numX=len(x_bands),
                    numY=len(Y_BANDS),
                )
            )
        )


        # ------------------------------------------------------------
        # COEFFICIENTS
        # ------------------------------------------------------------

        coeffs = (
            regression
            .select(
                "coefficients"
            )
            .arrayFlatten([
                x_bands,
                Y_BANDS,
            ])
            .toFloat()
        )


        # ------------------------------------------------------------
        # RMSE
        #
        # Preserve the same residual handling used in the 2018
        # K1-K4 sensitivity script.
        # ------------------------------------------------------------

        rmse = (
            regression
            .select(
                "residuals"
            )
            .arrayFlatten([
                Y_BANDS
            ])
            .rename(
                [
                    f"RMSE_{band}"
                    for band in Y_BANDS
                ]
            )
            .toFloat()
        )


        # ------------------------------------------------------------
        # FINAL STACK
        # ------------------------------------------------------------

        stack = (
            coeffs
            .addBands(
                nobs.toFloat()
            )
            .addBands(
                rmse
            )
            .toFloat()
        )


        if APPLY_MIN_NOBS_MASK:

            stack = (
                stack
                .updateMask(
                    fit_mask
                )
            )


        # ------------------------------------------------------------
        # MODEL METADATA
        # ------------------------------------------------------------

        stack = stack.set({
            "year":
                year,

            "harmonic_order":
                order,

            "fundamental_period_months":
                12,

            "shortest_period_months":
                12 / order,

            "n_response_variables":
                len(Y_BANDS),

            "n_predictors":
                len(x_bands),
        })


        # ------------------------------------------------------------
        # BAND-COUNT QA
        #
        # K2:
        #   5 predictors * 15 variables
        #   + nobs
        #   + 15 RMSE
        #   = 91
        #
        # K3:
        #   7 predictors * 15 variables
        #   + nobs
        #   + 15 RMSE
        #   = 121
        # ------------------------------------------------------------

        expected_bands = (
            len(x_bands)
            * len(Y_BANDS)
            + 1
            + len(Y_BANDS)
        )

        actual_bands = (
            stack
            .bandNames()
            .size()
            .getInfo()
        )


        print(
            f"K{order} predictors:",
            x_bands
        )

        print(
            f"K{order} output bands:",
            actual_bands,
            f"(expected {expected_bands})"
        )


        assert (
            actual_bands
            == expected_bands
        ), (
            f"K{order} band-count mismatch: "
            f"{actual_bands} != {expected_bands}"
        )


        # ------------------------------------------------------------
        # SAMPLE ONLY THE FIELD PLOTS
        #
        # No raster export at this stage.
        # ------------------------------------------------------------

        sampled = (
            stack
            .sampleRegions(
                collection=year_points,
                properties=[
                    "PlotID",
                    "Year",
                ],
                scale=TARGET_SCALE,
                projection=ee.Projection(
                    TARGET_CRS
                ),
                geometries=False,
                tileScale=4,
            )
        )


        key = (
            f"{year}_K{order}"
        )

        sampled_tables[key] = sampled


        print(
            f"{key} sampled rows:",
            sampled.size().getInfo()
        )


YEAR 2016
Plots: 378
Processed daily observations: 76

Fitting K2 ...
K2 predictors: ['constant', 'sin1', 'cos1', 'sin2', 'cos2']
K2 output bands: 91 (expected 91)
2016_K2 sampled rows: 378

Fitting K3 ...
K3 predictors: ['constant', 'sin1', 'cos1', 'sin2', 'cos2', 'sin3', 'cos3']
K3 output bands: 121 (expected 121)
2016_K3 sampled rows: 378

YEAR 2025
Plots: 118
Processed daily observations: 149

Fitting K2 ...
K2 predictors: ['constant', 'sin1', 'cos1', 'sin2', 'cos2']
K2 output bands: 91 (expected 91)
2025_K2 sampled rows: 118

Fitting K3 ...
K3 predictors: ['constant', 'sin1', 'cos1', 'sin2', 'cos2', 'sin3', 'cos3']
K3 output bands: 121 (expected 121)
2025_K3 sampled rows: 118


In [90]:
# ---------------------------------------------------------------------
# CELL 17
# CONVERT CELL 16 OUTPUT TO THE SAME FORMAT AS THE EXISTING
# 2018 *_PlotLevel_InterceptAmpPhase.csv FILES
#
# Output schema:
#
# PlotID
# Year
# HarmonicOrder
# X
# Y
# constant_*
# amp1_*
# phase1_*
# amp2_*
# phase2_*
# [amp3_* / phase3_* for K3]
# nobs
# RMSE_*
#
# ---------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# OUTPUT DIRECTORY
# ---------------------------------------------------------------------

EXPORT_DIR = Path(
    r"A:\NCA_DATA\Validation\Information_Partitioning"
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# MASTER POINT TABLE
#
# Used to restore X / Y, since Cell 16 sampled only PlotID + Year.
# ---------------------------------------------------------------------

MASTER_POINT_CSV = Path(
    r"N:\Data02\projects-active\BOPclassification_2025"
    r"\Field Data (Processed)\2026"
    r"\MasterPointFeatures_2016to2026.csv"
)


points_lookup = pd.read_csv(
    MASTER_POINT_CSV
)


points_lookup["PlotID"] = normalize_plot_id(
    points_lookup["PlotID"]
)

points_lookup["Year"] = (
    pd.to_numeric(
        points_lookup["Year"],
        errors="raise",
    )
    .astype(int)
)


points_lookup = (
    points_lookup[
        [
            "PlotID",
            "Year",
            "X",
            "Y",
        ]
    ]
    .drop_duplicates(
        subset=[
            "PlotID",
            "Year",
        ]
    )
)


# ---------------------------------------------------------------------
# EARTH ENGINE FEATURECOLLECTION -> PANDAS
# ---------------------------------------------------------------------

def feature_collection_to_df(
    fc,
):

    info = fc.getInfo()

    rows = [
        feature["properties"]
        for feature in info["features"]
    ]

    return pd.DataFrame(
        rows
    )


# ---------------------------------------------------------------------
# RAW SIN/COS -> INTERCEPT / AMPLITUDE / PHASE
#
# amplitude:
#
#     sqrt(sin^2 + cos^2)
#
# phase:
#
#     atan2(sin, cos)
#
# wrapped to [0, 2*pi)
#
# ---------------------------------------------------------------------

def to_intercept_amp_phase(
    raw_df,
    year,
    order,
):

    df = raw_df.copy()


    # -------------------------------------------------------------
    # IDENTIFIERS
    # -------------------------------------------------------------

    df["PlotID"] = normalize_plot_id(
        df["PlotID"]
    )

    df["Year"] = (
        pd.to_numeric(
            df["Year"],
            errors="raise",
        )
        .astype(int)
    )


    # -------------------------------------------------------------
    # RESTORE X / Y
    # -------------------------------------------------------------

    df = df.merge(
        points_lookup,
        on=[
            "PlotID",
            "Year",
        ],
        how="left",
        validate="one_to_one",
    )


    if (
        df["X"].isna().any()
        or
        df["Y"].isna().any()
    ):

        raise RuntimeError(
            f"{year}_K{order}: "
            "one or more plots could not be matched to X/Y."
        )


    # -------------------------------------------------------------
    # BASE OUTPUT COLUMNS
    # -------------------------------------------------------------

    out = pd.DataFrame({
        "PlotID":
            df["PlotID"],

        "Year":
            df["Year"],

        "HarmonicOrder":
            order,

        "X":
            df["X"],

        "Y":
            df["Y"],
    })


    # -------------------------------------------------------------
    # INTERCEPTS
    # -------------------------------------------------------------

    for band in Y_BANDS:

        out[
            f"constant_{band}"
        ] = df[
            f"constant_{band}"
        ]


    # -------------------------------------------------------------
    # AMPLITUDE + PHASE
    #
    # Keep exact 2018 column ordering:
    #
    # amp1 all bands
    # phase1 all bands
    # amp2 all bands
    # phase2 all bands
    # ...
    # -------------------------------------------------------------

    for harmonic in range(
        1,
        order + 1,
    ):

        # Amplitudes first
        for band in Y_BANDS:

            sin_col = (
                f"sin{harmonic}_{band}"
            )

            cos_col = (
                f"cos{harmonic}_{band}"
            )

            out[
                f"amp{harmonic}_{band}"
            ] = np.sqrt(
                df[sin_col] ** 2
                +
                df[cos_col] ** 2
            )


        # Then phases
        for band in Y_BANDS:

            sin_col = (
                f"sin{harmonic}_{band}"
            )

            cos_col = (
                f"cos{harmonic}_{band}"
            )

            phase = np.arctan2(
                df[sin_col],
                df[cos_col],
            )

            out[
                f"phase{harmonic}_{band}"
            ] = phase       


    # -------------------------------------------------------------
    # NOBS
    # -------------------------------------------------------------

    out["nobs"] = df["nobs"]


    # -------------------------------------------------------------
    # RMSE
    # -------------------------------------------------------------

    for band in Y_BANDS:

        out[
            f"RMSE_{band}"
        ] = df[
            f"RMSE_{band}"
        ]


    return out


# ---------------------------------------------------------------------
# BUILD + EXPORT ALL FOUR TABLES
# ---------------------------------------------------------------------

exported_tables = {}


for year in YEARS:

    for order in ORDERS:

        key = f"{year}_K{order}"

        print(
            "\n" + "=" * 60
        )

        print(
            "Preparing:",
            key
        )


        # ---------------------------------------------------------
        # DOWNLOAD POINT-SAMPLED RAW COEFFICIENTS
        # ---------------------------------------------------------

        raw_df = feature_collection_to_df(
            sampled_tables[key]
        )


        # ---------------------------------------------------------
        # DERIVE 2018-COMPATIBLE REPRESENTATION
        # ---------------------------------------------------------

        out = to_intercept_amp_phase(
            raw_df=raw_df,
            year=year,
            order=order,
        )


        # ---------------------------------------------------------
        # SORT
        # ---------------------------------------------------------

        out = (
            out
            .sort_values(
                [
                    "Year",
                    "PlotID",
                ]
            )
            .reset_index(
                drop=True
            )
        )


        # ---------------------------------------------------------
        # OUTPUT NAME
        #
        # Mirrors the existing 2018 naming convention.
        # ---------------------------------------------------------

        output_path = (
            EXPORT_DIR
            /
            (
                f"{year}_K{order}_"
                "PlotLevel_InterceptAmpPhase.csv"
            )
        )


        out.to_csv(
            output_path,
            index=False,
        )


        exported_tables[key] = out


        print(
            "Shape:",
            out.shape
        )

        print(
            "Unique PlotIDs:",
            out["PlotID"].nunique()
        )

        print(
            "Missing values:",
            int(
                out
                .isna()
                .sum()
                .sum()
            )
        )

        print(
            "Written:",
            output_path
        )


Preparing: 2016_K2
Shape: (378, 96)
Unique PlotIDs: 378
Missing values: 0
Written: A:\NCA_DATA\Validation\Information_Partitioning\2016_K2_PlotLevel_InterceptAmpPhase.csv

Preparing: 2016_K3


C:\Users\scottfordham\AppData\Local\Temp\ipykernel_40048\561135235.py:269: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[
C:\Users\scottfordham\AppData\Local\Temp\ipykernel_40048\561135235.py:269: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[
C:\Users\scottfordham\AppData\Local\Temp\ipykernel_40048\561135235.py:269: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1)

Shape: (378, 126)
Unique PlotIDs: 378
Missing values: 0
Written: A:\NCA_DATA\Validation\Information_Partitioning\2016_K3_PlotLevel_InterceptAmpPhase.csv

Preparing: 2025_K2
Shape: (118, 96)
Unique PlotIDs: 118
Missing values: 0
Written: A:\NCA_DATA\Validation\Information_Partitioning\2025_K2_PlotLevel_InterceptAmpPhase.csv

Preparing: 2025_K3
Shape: (118, 126)
Unique PlotIDs: 118
Missing values: 0
Written: A:\NCA_DATA\Validation\Information_Partitioning\2025_K3_PlotLevel_InterceptAmpPhase.csv


C:\Users\scottfordham\AppData\Local\Temp\ipykernel_40048\561135235.py:269: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[
C:\Users\scottfordham\AppData\Local\Temp\ipykernel_40048\561135235.py:269: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[
C:\Users\scottfordham\AppData\Local\Temp\ipykernel_40048\561135235.py:269: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1)

In [91]:
# ---------------------------------------------------------------------
# RERUN CORRECTED 2016 K2 + K3 HARMONIC POINT MODELS
#
# Input points:
#   2016_VegetationValidationPoints_EPSG26911.csv
#
# Sentinel-2 input:
#   existing prepared_years[2016]["collection"]
#
# Output:
#   2016_K2_PlotLevel_InterceptAmpPhase.csv
#   2016_K3_PlotLevel_InterceptAmpPhase.csv
#
# Both K2 and K3 are independently refit from the same processed
# 2016 Sentinel-2 observations.
# ---------------------------------------------------------------------

import ee

import numpy as np
import pandas as pd

VALIDATION_DIR = Path(
    r"A:\NCA_DATA\Validation\Information_Partitioning"
)


YEAR = 2016

POINTS_2016_CSV = (
    VALIDATION_DIR
    / "2016_VegetationValidationPoints_EPSG26911.csv"
)


# =====================================================================
# 1. READ CORRECTED 2016 POINT LAYER
# =====================================================================

points_df = pd.read_csv(
    POINTS_2016_CSV
)


points_df["PlotID"] = (
    normalize_plot_id(
        points_df["PlotID"]
    )
)

points_df["Year"] = (
    pd.to_numeric(
        points_df["Year"],
        errors="raise",
    )
    .astype(int)
)


assert (
    points_df["Year"]
    .eq(YEAR)
    .all()
)

assert (
    points_df["PlotID"]
    .is_unique
)


print(
    "Corrected 2016 points:",
    len(points_df)
)


# =====================================================================
# 2. LOCAL POINT TABLE -> EARTH ENGINE FEATURECOLLECTION
#
# Coordinates are already EPSG:26911.
# =====================================================================

features = []


for row in points_df.itertuples(
    index=False
):

    geom = ee.Geometry.Point(
        [
            float(row.X),
            float(row.Y),
        ],
        proj=TARGET_CRS,
    )

    feature = ee.Feature(
        geom,
        {
            "PlotID":
                str(row.PlotID),

            "Year":
                int(row.Year),

            "X":
                float(row.X),

            "Y":
                float(row.Y),
        },
    )

    features.append(
        feature
    )


year_points = ee.FeatureCollection(
    features
)


print(
    "Earth Engine points:",
    year_points.size().getInfo()
)


# =====================================================================
# 3. EXISTING PREPARED 2016 DAILY SENTINEL-2 COLLECTION
# =====================================================================

processed = (
    prepared_years[YEAR][
        "collection"
    ]
)


print(
    "Processed daily observations:",
    processed.size().getInfo()
)


# =====================================================================
# 4. HELPER:
#    EARTH ENGINE FEATURECOLLECTION -> PANDAS
# =====================================================================

def feature_collection_to_df(
    fc,
):

    info = fc.getInfo()

    rows = [
        feature["properties"]
        for feature in info["features"]
    ]

    return pd.DataFrame(
        rows
    )


# =====================================================================
# 5. HELPER:
#    RAW SIN/COS COEFFICIENTS -> INTERCEPT / AMP / PHASE
#
# This builds the output as column blocks to avoid pandas
# fragmentation warnings.
# =====================================================================

def raw_to_intercept_amp_phase(
    raw_df,
    points_df,
    year,
    order,
):

    df = raw_df.copy()


    # -------------------------------------------------------------
    # NORMALIZE IDENTIFIERS
    # -------------------------------------------------------------

    df["PlotID"] = (
        normalize_plot_id(
            df["PlotID"]
        )
    )

    df["Year"] = (
        pd.to_numeric(
            df["Year"],
            errors="raise",
        )
        .astype(int)
    )


    # -------------------------------------------------------------
    # RESTORE AUTHORITATIVE X/Y FROM CORRECTED POINT TABLE
    # -------------------------------------------------------------

    xy = (
        points_df[
            [
                "PlotID",
                "Year",
                "X",
                "Y",
            ]
        ]
        .copy()
    )


    df = df.merge(
        xy,
        on=[
            "PlotID",
            "Year",
        ],
        how="left",
        validate="one_to_one",
    )


    if (
        df["X"].isna().any()
        or
        df["Y"].isna().any()
    ):

        raise RuntimeError(
            f"{year}_K{order}: "
            "missing X/Y after point-table join."
        )


    # -------------------------------------------------------------
    # BUILD OUTPUT DICTIONARY
    #
    # One DataFrame construction at the end avoids fragmentation.
    # -------------------------------------------------------------

    output = {
        "PlotID":
            df["PlotID"].to_numpy(),

        "Year":
            df["Year"].to_numpy(),

        "HarmonicOrder":
            np.full(
                len(df),
                order,
                dtype=int,
            ),

        "X":
            df["X"].to_numpy(),

        "Y":
            df["Y"].to_numpy(),
    }


    # -------------------------------------------------------------
    # INTERCEPTS
    # -------------------------------------------------------------

    for band in Y_BANDS:

        output[
            f"constant_{band}"
        ] = (
            df[
                f"constant_{band}"
            ]
            .to_numpy()
        )


    # -------------------------------------------------------------
    # AMPLITUDE + PHASE
    #
    # Preserve existing output order:
    #
    # amp1_*
    # phase1_*
    # amp2_*
    # phase2_*
    # [amp3_* / phase3_*]
    # -------------------------------------------------------------

    for harmonic in range(
        1,
        order + 1,
    ):

        # -------------------------
        # AMPLITUDE
        # -------------------------

        for band in Y_BANDS:

            sin_values = (
                df[
                    f"sin{harmonic}_{band}"
                ]
                .to_numpy()
            )

            cos_values = (
                df[
                    f"cos{harmonic}_{band}"
                ]
                .to_numpy()
            )

            output[
                f"amp{harmonic}_{band}"
            ] = np.sqrt(
                sin_values ** 2
                +
                cos_values ** 2
            )


        # -------------------------
        # PHASE
        # -------------------------

        for band in Y_BANDS:

            sin_values = (
                df[
                    f"sin{harmonic}_{band}"
                ]
                .to_numpy()
            )

            cos_values = (
                df[
                    f"cos{harmonic}_{band}"
                ]
                .to_numpy()
            )

            phase = np.arctan2(
                sin_values,
                cos_values,
            )

            output[
                f"phase{harmonic}_{band}"
            ] = phase


    # -------------------------------------------------------------
    # NOBS
    # -------------------------------------------------------------

    output["nobs"] = (
        df["nobs"]
        .to_numpy()
    )


    # -------------------------------------------------------------
    # RMSE
    # -------------------------------------------------------------

    for band in Y_BANDS:

        output[
            f"RMSE_{band}"
        ] = (
            df[
                f"RMSE_{band}"
            ]
            .to_numpy()
        )


    out = pd.DataFrame(
        output
    )


    return (
        out
        .sort_values(
            [
                "Year",
                "PlotID",
            ]
        )
        .reset_index(
            drop=True
        )
    )


# =====================================================================
# 6. FIT + SAMPLE K2 AND K3
# =====================================================================

corrected_2016_tables = {}


for order in [
    2,
    3,
]:

    print(
        "\n" + "=" * 60
    )

    print(
        f"Fitting corrected 2016 K{order}"
    )


    # -------------------------------------------------------------
    # DESIGN MATRIX
    # -------------------------------------------------------------

    x_bands = make_x_bands(
        order
    )


    # -------------------------------------------------------------
    # ADD TIME TERMS
    # -------------------------------------------------------------

    with_x = (
        processed
        .map(
            lambda img:
            add_time_terms_for_year(
                img,
                YEAR,
                order,
            )
        )
    )


    # -------------------------------------------------------------
    # NOBS
    # -------------------------------------------------------------

    nobs = (
        with_x
        .select("B2")
        .count()
        .round()
        .toInt16()
        .rename("nobs")
    )


    fit_mask = (
        nobs.gte(
            MIN_NOBS
        )
    )


    if APPLY_MIN_NOBS_MASK:

        with_x = (
            with_x
            .map(
                lambda img:
                ee.Image(img)
                .updateMask(
                    fit_mask
                )
            )
        )


    # -------------------------------------------------------------
    # MULTIVARIATE OLS
    # -------------------------------------------------------------

    regression = (
        with_x
        .select(
            x_bands
            +
            Y_BANDS
        )
        .reduce(
            ee.Reducer.linearRegression(
                numX=len(
                    x_bands
                ),
                numY=len(
                    Y_BANDS
                ),
            )
        )
    )


    # -------------------------------------------------------------
    # COEFFICIENTS
    # -------------------------------------------------------------

    coeffs = (
        regression
        .select(
            "coefficients"
        )
        .arrayFlatten([
            x_bands,
            Y_BANDS,
        ])
        .toFloat()
    )


    # -------------------------------------------------------------
    # RMSE
    # -------------------------------------------------------------

    rmse = (
        regression
        .select(
            "residuals"
        )
        .arrayFlatten([
            Y_BANDS
        ])
        .rename(
            [
                f"RMSE_{band}"
                for band in Y_BANDS
            ]
        )
        .toFloat()
    )


    # -------------------------------------------------------------
    # STACK
    # -------------------------------------------------------------

    stack = (
        coeffs
        .addBands(
            nobs.toFloat()
        )
        .addBands(
            rmse
        )
        .toFloat()
    )


    if APPLY_MIN_NOBS_MASK:

        stack = (
            stack
            .updateMask(
                fit_mask
            )
        )


    # -------------------------------------------------------------
    # BAND-COUNT QA
    # -------------------------------------------------------------

    expected_bands = (
        len(x_bands)
        *
        len(Y_BANDS)
        +
        1
        +
        len(Y_BANDS)
    )


    actual_bands = (
        stack
        .bandNames()
        .size()
        .getInfo()
    )


    print(
        "Predictors:",
        x_bands
    )

    print(
        "Output bands:",
        actual_bands,
        f"(expected {expected_bands})"
    )


    assert (
        actual_bands
        ==
        expected_bands
    )


    # -------------------------------------------------------------
    # SAMPLE CORRECTED 2016 POINT POPULATION
    # -------------------------------------------------------------

    sampled = (
        stack
        .sampleRegions(
            collection=year_points,

            properties=[
                "PlotID",
                "Year",
            ],

            scale=TARGET_SCALE,

            projection=ee.Projection(
                TARGET_CRS
            ),

            geometries=False,

            tileScale=4,
        )
    )


    sampled_n = (
        sampled
        .size()
        .getInfo()
    )


    print(
        "Sampled rows:",
        sampled_n
    )


    # -------------------------------------------------------------
    # DOWNLOAD
    # -------------------------------------------------------------

    raw_df = (
        feature_collection_to_df(
            sampled
        )
    )


    # -------------------------------------------------------------
    # DERIVE INTERCEPT / AMP / PHASE
    # -------------------------------------------------------------

    out = (
        raw_to_intercept_amp_phase(
            raw_df=raw_df,
            points_df=points_df,
            year=YEAR,
            order=order,
        )
    )


    # -------------------------------------------------------------
    # FINAL QA
    # -------------------------------------------------------------

    expected_columns = (
        96
        if order == 2
        else 126
    )


    print(
        "Final shape:",
        out.shape
    )

    print(
        "Unique PlotIDs:",
        out["PlotID"].nunique()
    )

    print(
        "Missing values:",
        int(
            out
            .isna()
            .sum()
            .sum()
        )
    )

    print(
        "nobs range:",
        (
            int(
                out["nobs"].min()
            ),
            int(
                out["nobs"].max()
            ),
        )
    )


    assert (
        len(out.columns)
        ==
        expected_columns
    )


    assert (
        out["PlotID"]
        .is_unique
    )


    assert (
        out
        .isna()
        .sum()
        .sum()
        ==
        0
    )


    # -------------------------------------------------------------
    # WRITE / REPLACE 2016 OUTPUT
    # -------------------------------------------------------------

    output_path = (
        VALIDATION_DIR
        /
        (
            f"2016_K{order}_"
            "PlotLevel_InterceptAmpPhase.csv"
        )
    )


    out.to_csv(
        output_path,
        index=False,
    )


    corrected_2016_tables[
        f"2016_K{order}"
    ] = out


    print(
        "Written:",
        output_path
    )


# =====================================================================
# 7. FINAL K2 / K3 POPULATION CONSISTENCY CHECK
# =====================================================================

k2_corrected = (
    corrected_2016_tables[
        "2016_K2"
    ]
)

k3_corrected = (
    corrected_2016_tables[
        "2016_K3"
    ]
)


assert (
    set(
        k2_corrected["PlotID"]
    )
    ==
    set(
        k3_corrected["PlotID"]
    )
), (
    "Corrected 2016 K2 and K3 do not contain "
    "the same PlotIDs."
)


print(
    "\n" + "=" * 60
)

print(
    "CORRECTED 2016 HARMONIC EXPORT COMPLETE"
)

print(
    "K2:",
    k2_corrected.shape
)

print(
    "K3:",
    k3_corrected.shape
)

print(
    "Shared plots:",
    len(k2_corrected)
)

Corrected 2016 points: 361
Earth Engine points: 361
Processed daily observations: 76

Fitting corrected 2016 K2
Predictors: ['constant', 'sin1', 'cos1', 'sin2', 'cos2']
Output bands: 91 (expected 91)
Sampled rows: 257
Final shape: (257, 96)
Unique PlotIDs: 257
Missing values: 0
nobs range: (16, 37)
Written: A:\NCA_DATA\Validation\Information_Partitioning\2016_K2_PlotLevel_InterceptAmpPhase.csv

Fitting corrected 2016 K3
Predictors: ['constant', 'sin1', 'cos1', 'sin2', 'cos2', 'sin3', 'cos3']
Output bands: 121 (expected 121)
Sampled rows: 257
Final shape: (257, 126)
Unique PlotIDs: 257
Missing values: 0
nobs range: (16, 37)
Written: A:\NCA_DATA\Validation\Information_Partitioning\2016_K3_PlotLevel_InterceptAmpPhase.csv

CORRECTED 2016 HARMONIC EXPORT COMPLETE
K2: (257, 96)
K3: (257, 126)
Shared plots: 257


In [83]:
# ---------------------------------------------------------------------
# DIAGNOSE WHY 104 CORRECTED 2016 POINTS ARE DROPPED
# ---------------------------------------------------------------------

# Use the same corrected 2016 point FeatureCollection:
# year_points

# Use the same processed 2016 daily collection:
# processed


# ---------------------------------------------------------------------
# 1. B2 NOBS AT ALL 361 POINTS
# ---------------------------------------------------------------------

nobs_2016 = (
    processed
    .select("B2")
    .count()
    .rename("nobs")
)


nobs_sample = (
    nobs_2016
    .sampleRegions(
        collection=year_points,
        properties=[
            "PlotID",
            "Year",
        ],
        scale=TARGET_SCALE,
        projection=ee.Projection(
            TARGET_CRS
        ),
        geometries=False,
        tileScale=4,
    )
)


nobs_info = nobs_sample.getInfo()


nobs_df = pd.DataFrame(
    [
        feature["properties"]
        for feature in nobs_info["features"]
    ]
)


nobs_df["PlotID"] = (
    normalize_plot_id(
        nobs_df["PlotID"]
    )
)


print(
    "Corrected input points:",
    len(points_df)
)

print(
    "Points with nobs value:",
    len(nobs_df)
)


print(
    "\nNOBS distribution:"
)

print(
    nobs_df["nobs"]
    .describe()
)


print(
    "\nPlots nobs >= 12:",
    int(
        nobs_df["nobs"]
        .ge(MIN_NOBS)
        .sum()
    )
)

print(
    "Plots nobs < 12:",
    int(
        nobs_df["nobs"]
        .lt(MIN_NOBS)
        .sum()
    )
)


# ---------------------------------------------------------------------
# 2. WHICH PLOTS FAIL NOBS?
# ---------------------------------------------------------------------

low_nobs = (
    nobs_df.loc[
        nobs_df["nobs"].lt(
            MIN_NOBS
        ),
        [
            "PlotID",
            "Year",
            "nobs",
        ]
    ]
    .sort_values(
        [
            "nobs",
            "PlotID",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nPlots below MIN_NOBS:",
    len(low_nobs)
)


if len(low_nobs) > 0:

    display(
        low_nobs
    )


# ---------------------------------------------------------------------
# 3. COMPARE WITH ACTUALLY RETURNED K2 PLOTS
# ---------------------------------------------------------------------

returned_k2_ids = set(
    normalize_plot_id(
        raw_df["PlotID"]
    )
)


all_point_ids = set(
    points_df["PlotID"]
)


dropped_k2_ids = (
    all_point_ids
    -
    returned_k2_ids
)


print(
    "\nK2 plots returned:",
    len(returned_k2_ids)
)

print(
    "K2 plots dropped:",
    len(dropped_k2_ids)
)


# ---------------------------------------------------------------------
# 4. CROSS-TAB DROP REASON
# ---------------------------------------------------------------------

drop_diagnostic = (
    points_df[
        [
            "PlotID",
            "X",
            "Y",
        ]
    ]
    .merge(
        nobs_df[
            [
                "PlotID",
                "nobs",
            ]
        ],
        on="PlotID",
        how="left",
        validate="one_to_one",
    )
)


drop_diagnostic["K2_returned"] = (
    drop_diagnostic["PlotID"]
    .isin(
        returned_k2_ids
    )
)


drop_diagnostic["nobs_pass"] = (
    drop_diagnostic["nobs"]
    .ge(
        MIN_NOBS
    )
)


print(
    "\nDrop diagnostic:"
)

print(
    pd.crosstab(
        drop_diagnostic["nobs_pass"],
        drop_diagnostic["K2_returned"],
        margins=True,
    )
)


# Points with enough B2 observations
# but still absent from the regression output
unexpected_drops = (
    drop_diagnostic.loc[
        drop_diagnostic["nobs_pass"]
        &
        ~drop_diagnostic["K2_returned"]
    ]
    .sort_values(
        "PlotID"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nAdequate nobs but missing from K2:",
    len(unexpected_drops)
)


if len(unexpected_drops) > 0:

    display(
        unexpected_drops
    )

Corrected input points: 361
Points with nobs value: 257

NOBS distribution:
count    257.000000
mean      28.782101
std        5.323077
min       16.000000
25%       28.000000
50%       31.000000
75%       32.000000
max       37.000000
Name: nobs, dtype: float64

Plots nobs >= 12: 257
Plots nobs < 12: 0

Plots below MIN_NOBS: 0

K2 plots returned: 257
K2 plots dropped: 104

Drop diagnostic:
K2_returned  False  True  All
nobs_pass                    
False          104     0  104
True             0   257  257
All            104   257  361

Adequate nobs but missing from K2: 0


In [84]:
# ---------------------------------------------------------------------
# DIAGNOSE 104 POINTS WITH NO NOBS
# ---------------------------------------------------------------------

no_nobs = (
    drop_diagnostic.loc[
        drop_diagnostic["nobs"].isna()
    ]
    .copy()
)


print(
    "Points with no nobs:",
    len(no_nobs)
)

display(
    no_nobs[
        [
            "PlotID",
            "X",
            "Y",
        ]
    ]
)


# ---------------------------------------------------------------------
# PROCESSED 2016 COLLECTION FOOTPRINT
# ---------------------------------------------------------------------

processed_footprint = (
    processed
    .geometry()
)


# Build EE feature collection for the no-nobs points
no_nobs_features = []


for row in no_nobs.itertuples(
    index=False
):

    geom = ee.Geometry.Point(
        [
            float(row.X),
            float(row.Y),
        ],
        proj=TARGET_CRS,
    )

    no_nobs_features.append(
        ee.Feature(
            geom,
            {
                "PlotID": str(row.PlotID),
            },
        )
    )


no_nobs_fc = ee.FeatureCollection(
    no_nobs_features
)


# Mark whether each point intersects the processed collection footprint
def mark_inside_footprint(feature):

    feature = ee.Feature(
        feature
    )

    inside = processed_footprint.contains(
        feature.geometry(),
        maxError=1,
    )

    return feature.set(
        "inside_processed_footprint",
        inside,
    )


footprint_check = (
    no_nobs_fc
    .map(
        mark_inside_footprint
    )
)


footprint_info = (
    footprint_check
    .aggregate_array(
        "inside_processed_footprint"
    )
    .getInfo()
)


print(
    "Inside processed footprint:",
    sum(footprint_info)
)

print(
    "Outside processed footprint:",
    len(footprint_info)
    -
    sum(footprint_info)
)

Points with no nobs: 104


,PlotID,X,Y
52,300,609655.772,4734725.079
53,301,611237.164,4734685.254
54,302,611049.988,4735006.138
57,305,618260.703,4723053.545
58,306,618174.513,4723180.309
...,...,...,...
166,700,629190.991,4698049.681
167,701,629085.919,4698097.872
168,702,631327.153,4699099.838
169,703,631383.738,4699108.141


Inside processed footprint: 104
Outside processed footprint: 0


In [87]:
# ---------------------------------------------------------------------
# ROBUST RAW 2016 B2 COVERAGE DIAGNOSTIC
#
# Purpose:
#   Determine whether the 104 no-nobs points have any valid B2 pixels
#   in the RAW custom 2016 Sentinel-2 collection.
#
# Unlike sampleRegions(), this preserves every input point and writes
# raw_nobs = 0 when no valid B2 observations are present.
# ---------------------------------------------------------------------

YEAR = 2016


# ---------------------------------------------------------------------
# RAW CUSTOM HISTORICAL COLLECTION
# ---------------------------------------------------------------------

raw_2016 = (
    ee.ImageCollection(
        "projects/bop-nca-data-space/assets/S2_C1_L2A_2016_2017"
    )
    .filterDate(
        f"{YEAR}-01-01",
        f"{YEAR + 1}-01-01",
    )
)


print(
    "Raw 2016 source images:",
    raw_2016.size().getInfo()
)


# ---------------------------------------------------------------------
# RAW B2 VALID-OBSERVATION COUNT IMAGE
# ---------------------------------------------------------------------

raw_b2_count = (
    raw_2016
    .select("B2")
    .count()
    .rename("raw_nobs")
)


# ---------------------------------------------------------------------
# BUILD EE FEATURECOLLECTION FROM THE 104 NO-NOBS POINTS
# ---------------------------------------------------------------------

no_nobs_features = []


for row in no_nobs.itertuples(
    index=False
):

    no_nobs_features.append(
        ee.Feature(
            ee.Geometry.Point(
                [
                    float(row.X),
                    float(row.Y),
                ],
                proj=TARGET_CRS,
            ),
            {
                "PlotID": str(row.PlotID),
                "X": float(row.X),
                "Y": float(row.Y),
            },
        )
    )


no_nobs_fc = ee.FeatureCollection(
    no_nobs_features
)


print(
    "No-nobs points:",
    no_nobs_fc.size().getInfo()
)


# ---------------------------------------------------------------------
# ATTACH RAW NOBS TO EVERY POINT
#
# reduceRegion returns null if the count image is masked at the point.
# Convert null -> 0 explicitly.
# ---------------------------------------------------------------------

def attach_raw_nobs(feature):

    feature = ee.Feature(
        feature
    )

    value = (
        raw_b2_count
        .reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=feature.geometry(),
            scale=TARGET_SCALE,
            crs=TARGET_CRS,
            maxPixels=1e6,
        )
        .get("raw_nobs")
    )

    raw_nobs = ee.Number(
        ee.Algorithms.If(
            ee.Algorithms.IsEqual(
                value,
                None,
            ),
            0,
            value,
        )
    )

    return feature.set(
        "raw_nobs",
        raw_nobs,
    )


raw_point_check = (
    no_nobs_fc
    .map(
        attach_raw_nobs
    )
)


# ---------------------------------------------------------------------
# DOWNLOAD ALL 104 RESULTS
# ---------------------------------------------------------------------

raw_info = (
    raw_point_check
    .getInfo()
)


raw_count_df = pd.DataFrame(
    [
        feature["properties"]
        for feature in raw_info["features"]
    ]
)


raw_count_df["PlotID"] = (
    normalize_plot_id(
        raw_count_df["PlotID"]
    )
)


raw_count_df["raw_nobs"] = (
    pd.to_numeric(
        raw_count_df["raw_nobs"],
        errors="coerce",
    )
    .fillna(0)
    .astype(int)
)


# ---------------------------------------------------------------------
# QA
# ---------------------------------------------------------------------

assert (
    len(raw_count_df)
    ==
    len(no_nobs)
), (
    "Raw diagnostic did not return all no-nobs points."
)


print(
    "\nRaw B2 diagnostic:"
)

print(
    "Points tested:",
    len(raw_count_df)
)

print(
    "Raw B2 present:",
    int(
        raw_count_df["raw_nobs"]
        .gt(0)
        .sum()
    )
)

print(
    "Raw B2 absent:",
    int(
        raw_count_df["raw_nobs"]
        .eq(0)
        .sum()
    )
)


print(
    "\nRaw nobs summary:"
)

print(
    raw_count_df["raw_nobs"]
    .describe()
)


# ---------------------------------------------------------------------
# JOIN BACK TO LOCAL COORDINATES
# ---------------------------------------------------------------------

diagnostic_104 = (
    no_nobs[
        [
            "PlotID",
            "X",
            "Y",
        ]
    ]
    .merge(
        raw_count_df[
            [
                "PlotID",
                "raw_nobs",
            ]
        ],
        on="PlotID",
        how="left",
        validate="one_to_one",
    )
)


diagnostic_104["raw_has_data"] = (
    diagnostic_104["raw_nobs"]
    .gt(0)
)


display(
    diagnostic_104
    .sort_values(
        [
            "raw_has_data",
            "PlotID",
        ]
    )
)

Raw 2016 source images: 202
No-nobs points: 104

Raw B2 diagnostic:
Points tested: 104
Raw B2 present: 0
Raw B2 absent: 104

Raw nobs summary:
count    104.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: raw_nobs, dtype: float64


,PlotID,X,Y,raw_nobs,raw_has_data
0,300,609655.772,4734725.079,0,False
1,301,611237.164,4734685.254,0,False
2,302,611049.988,4735006.138,0,False
3,305,618260.703,4723053.545,0,False
4,306,618174.513,4723180.309,0,False
...,...,...,...,...,...
99,700,629190.991,4698049.681,0,False
100,701,629085.919,4698097.872,0,False
101,702,631327.153,4699099.838,0,False
102,703,631383.738,4699108.141,0,False


Below is QA60 diagnostic script to identify where and how to resolve all 2016 images getting masked. 

In [86]:
# ---------------------------------------------------------------------
# DIAGNOSE 2016 VALID OBSERVATIONS AT FIELD PLOTS
# ---------------------------------------------------------------------

year = 2016

year_points = (
    prepared_years[year]["points"]
)

processed = (
    prepared_years[year]["collection"]
)


# ------------------------------------------------------------
# COUNT VALID B2 OBSERVATIONS
# AFTER ALL 2016 PREPROCESSING
# ------------------------------------------------------------

nobs_img = (
    processed
    .select("B2")
    .count()
    .rename("nobs")
)


nobs_points = (
    nobs_img
    .sampleRegions(
        collection=year_points,
        properties=[
            "PlotID",
            "Year",
        ],
        scale=TARGET_SCALE,
        projection=ee.Projection(
            TARGET_CRS
        ),
        geometries=False,
        tileScale=4,
    )
)


print(
    "Input plots:",
    year_points.size().getInfo()
)

print(
    "Plots with an nobs value:",
    nobs_points.size().getInfo()
)


print(
    "\nNOBS statistics:"
)

print(
    nobs_points
    .aggregate_stats(
        "nobs"
    )
    .getInfo()
)


print(
    "\nPlots nobs >= 12:",
    nobs_points
    .filter(
        ee.Filter.gte(
            "nobs",
            MIN_NOBS,
        )
    )
    .size()
    .getInfo()
)


print(
    "Plots nobs < 12:",
    nobs_points
    .filter(
        ee.Filter.lt(
            "nobs",
            MIN_NOBS,
        )
    )
    .size()
    .getInfo()
)

Input plots: 378
Plots with an nobs value: 378

NOBS statistics:
{'max': 37, 'mean': 29.296296296296298, 'min': 16, 'sample_sd': 3.6223645188925495, 'sample_var': 13.12152470773165, 'sum': 11074, 'sum_sq': 329374, 'total_count': 378, 'total_sd': 3.6175698582618026, 'total_var': 13.086811679404317, 'valid_count': 378, 'weight_sum': 378, 'weighted_sum': 11074}

Plots nobs >= 12: 378
Plots nobs < 12: 0


In [78]:
# ---------------------------------------------------------------------
# DIAGNOSE WHERE 2016 POINT PIXELS DISAPPEAR
# ---------------------------------------------------------------------

year = 2016

year_points = (
    point_fc
    .filter(
        ee.Filter.eq(
            "Year",
            year,
        )
    )
)

source = get_year_collection(
    year,
    year_points,
)


# ------------------------------------------------------------
# BUILD EACH PREPROCESSING STAGE
# ------------------------------------------------------------

raw = source

scaled = (
    raw
    .map(scale_sr)
)

cloud_masked = (
    scaled
    .map(mask_clouds_shadows)
)

selected = (
    cloud_masked
    .map(select_bands)
)

daily = (
    dedupe_by_date(
        selected
    )
)


stages = {
    "raw":
        raw,

    "scaled":
        scaled,

    "cloud_masked":
        cloud_masked,

    "selected":
        selected,

    "daily":
        daily,
}


# ------------------------------------------------------------
# FOR EACH STAGE:
#
# 1. Count valid B2 observations per pixel
# 2. Sample that count at the 378 field plots
# ------------------------------------------------------------

for name, col in stages.items():

    print(
        "\n" + "-" * 60
    )

    print(
        name.upper()
    )

    print(
        "Collection images:",
        col.size().getInfo()
    )


    nobs_img = (
        col
        .select("B2")
        .count()
        .rename("nobs")
    )


    sampled = (
        nobs_img
        .sampleRegions(
            collection=year_points,
            properties=[
                "PlotID",
                "Year",
            ],
            scale=TARGET_SCALE,
            projection=ee.Projection(
                TARGET_CRS
            ),
            geometries=False,
            tileScale=4,
        )
    )


    n_sampled = (
        sampled.size().getInfo()
    )


    print(
        "Plots returned:",
        n_sampled
    )


    if n_sampled > 0:

        print(
            "NOBS stats:",
            sampled
            .aggregate_stats(
                "nobs"
            )
            .getInfo()
        )


------------------------------------------------------------
RAW
Collection images: 202
Plots returned: 378
NOBS stats: {'max': 126, 'mean': 72.1931216931217, 'min': 63, 'sample_sd': 21.280550024078945, 'sample_var': 452.8618093273264, 'sum': 27289, 'sum_sq': 2140807, 'total_count': 378, 'total_sd': 21.252382506738012, 'total_var': 451.66376221270383, 'valid_count': 378, 'weight_sum': 378, 'weighted_sum': 27289}

------------------------------------------------------------
SCALED
Collection images: 202
Plots returned: 378
NOBS stats: {'max': 126, 'mean': 72.1931216931217, 'min': 63, 'sample_sd': 21.280550024078945, 'sample_var': 452.8618093273264, 'sum': 27289, 'sum_sq': 2140807, 'total_count': 378, 'total_sd': 21.252382506738012, 'total_var': 451.66376221270383, 'valid_count': 378, 'weight_sum': 378, 'weighted_sum': 27289}

------------------------------------------------------------
CLOUD_MASKED
Collection images: 202
Plots returned: 378
NOBS stats: {'max': 65, 'mean': 33.0925925925

In [79]:
# ---------------------------------------------------------------------
# DIAGNOSE 2016 CLOUD MASK COMPONENTS
#
# Test:
#   raw
#   QA60 only
#   SCL only
#   cloud probability only
#   SCL + cloud probability
#   QA60 + SCL
#   QA60 + SCL + cloud probability
# ---------------------------------------------------------------------

year = 2016

year_points = (
    point_fc
    .filter(
        ee.Filter.eq(
            "Year",
            year,
        )
    )
)

source = get_year_collection(
    year,
    year_points,
)

scaled = source.map(
    scale_sr
)


# ---------------------------------------------------------------------
# INDIVIDUAL MASK FUNCTIONS
# ---------------------------------------------------------------------

def mask_qa60_only(img):

    img = ee.Image(img)

    qa = img.select(
        "QA60"
    )

    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11

    qa_mask = (
        qa
        .bitwiseAnd(cloud_bit)
        .eq(0)
        .And(
            qa
            .bitwiseAnd(cirrus_bit)
            .eq(0)
        )
    )

    return img.updateMask(
        qa_mask
    )


def mask_scl_only(img):

    img = ee.Image(img)

    scl = img.select(
        "SCL"
    )

    scl_mask = (
        scl
        .neq(0)
        .And(scl.neq(1))
        .And(scl.neq(3))
        .And(scl.neq(7))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    return img.updateMask(
        scl_mask
    )


def mask_cp_only(img):

    img = ee.Image(img)

    cp_matched = ee.Number(
        img.get(
            "cp_matched"
        )
    )

    cp_mask = ee.Image(
        ee.Algorithms.If(

            cp_matched.eq(1),

            img
            .select(
                "cloudProb"
            )
            .lt(
                CLOUD_PROB_THRESH
            ),

            ee.Image.constant(
                1
            ),
        )
    )

    return img.updateMask(
        cp_mask
    )


def mask_scl_cp(img):

    return mask_cp_only(
        mask_scl_only(
            img
        )
    )


def mask_qa_scl(img):

    return mask_scl_only(
        mask_qa60_only(
            img
        )
    )


def mask_all_three(img):

    return mask_cp_only(
        mask_scl_only(
            mask_qa60_only(
                img
            )
        )
    )


# ---------------------------------------------------------------------
# BUILD TEST COLLECTIONS
# ---------------------------------------------------------------------

tests = {

    "RAW_SCALED":
        scaled,

    "QA60_ONLY":
        scaled.map(
            mask_qa60_only
        ),

    "SCL_ONLY":
        scaled.map(
            mask_scl_only
        ),

    "CLOUD_PROB_ONLY":
        scaled.map(
            mask_cp_only
        ),

    "SCL_PLUS_CP":
        scaled.map(
            mask_scl_cp
        ),

    "QA60_PLUS_SCL":
        scaled.map(
            mask_qa_scl
        ),

    "QA60_PLUS_SCL_PLUS_CP":
        scaled.map(
            mask_all_three
        ),
}


# ---------------------------------------------------------------------
# SAMPLE VALID B2 OBSERVATION COUNTS AT FIELD POINTS
# ---------------------------------------------------------------------

for name, col in tests.items():

    print(
        "\n" + "-" * 60
    )

    print(
        name
    )

    nobs = (
        col
        .select("B2")
        .count()
        .rename("nobs")
    )

    sampled = (
        nobs
        .sampleRegions(
            collection=year_points,
            properties=[
                "PlotID",
                "Year",
            ],
            scale=TARGET_SCALE,
            projection=ee.Projection(
                TARGET_CRS
            ),
            geometries=False,
            tileScale=4,
        )
    )

    n_returned = (
        sampled
        .size()
        .getInfo()
    )

    print(
        "Plots returned:",
        n_returned
    )

    if n_returned > 0:

        print(
            "NOBS stats:",
            sampled
            .aggregate_stats(
                "nobs"
            )
            .getInfo()
        )


------------------------------------------------------------
RAW_SCALED
Plots returned: 378
NOBS stats: {'max': 126, 'mean': 72.1931216931217, 'min': 63, 'sample_sd': 21.280550024078945, 'sample_var': 452.8618093273264, 'sum': 27289, 'sum_sq': 2140807, 'total_count': 378, 'total_sd': 21.252382506738012, 'total_var': 451.66376221270383, 'valid_count': 378, 'weight_sum': 378, 'weighted_sum': 27289}

------------------------------------------------------------
QA60_ONLY
Plots returned: 378
NOBS stats: {'max': 74, 'mean': 41.32539682539682, 'min': 30, 'sample_sd': 12.637254531612232, 'sample_var': 159.70020209675388, 'sum': 15621, 'sum_sq': 705751, 'total_count': 378, 'total_sd': 12.620527516297392, 'total_var': 159.2777147896196, 'valid_count': 378, 'weight_sum': 378, 'weighted_sum': 15621}

------------------------------------------------------------
SCL_ONLY
Plots returned: 378
NOBS stats: {'max': 71, 'mean': 36.15343915343915, 'min': 16, 'sample_sd': 11.610740080764803, 'sample_var': 

In [ ]:
# ---------------------------------------------------------------------
# DIAGNOSE EARTH ENGINE QA60 INGESTION
#
# We want to distinguish:
#
#   1. QA60 VALUE problem
#   2. QA60 BAND MASK problem
#   3. QA60 bit-test problem
#
# Do this BEFORE updateMask().
# ---------------------------------------------------------------------

year = 2016

year_points = (
    point_fc
    .filter(
        ee.Filter.eq(
            "Year",
            year,
        )
    )
)

source = get_year_collection(
    year,
    year_points,
)


# ---------------------------------------------------------------------
# BUILD DIAGNOSTIC QA60 BANDS
# ---------------------------------------------------------------------

def qa60_diagnostics(img):

    img = ee.Image(img)

    qa = (
        img
        .select("QA60")
        .rename("QA60_raw")
    )


    # Intrinsic EE mask attached to QA60 itself.
    #
    # 1 = QA60 pixel exists
    # 0 = QA60 pixel is masked / nodata in EE

    qa_valid = (
        qa
        .mask()
        .rename("QA60_valid")
    )


    # Decode individual bits.

    opaque = (
        qa
        .bitwiseAnd(1 << 10)
        .neq(0)
        .rename("QA60_opaque")
    )

    cirrus = (
        qa
        .bitwiseAnd(1 << 11)
        .neq(0)
        .rename("QA60_cirrus")
    )


    # Normal clear-pixel test.

    clear = (
        qa
        .bitwiseAnd(1 << 10)
        .eq(0)
        .And(
            qa
            .bitwiseAnd(1 << 11)
            .eq(0)
        )
        .rename("QA60_clear")
    )


    return (
        qa
        .addBands(qa_valid)
        .addBands(opaque)
        .addBands(cirrus)
        .addBands(clear)
        .copyProperties(
            img,
            img.propertyNames(),
        )
    )


qa_diag = source.map(
    qa60_diagnostics
)


# ---------------------------------------------------------------------
# REDUCE ACROSS ALL 2016 IMAGES
#
# Count how often each plot has:
#
#   valid QA60
#   clear QA60
#   opaque cloud
#   cirrus
# ---------------------------------------------------------------------

qa_valid_count = (
    qa_diag
    .select("QA60_valid")
    .sum()
    .rename("qa_valid_n")
)

qa_clear_count = (
    qa_diag
    .select("QA60_clear")
    .sum()
    .rename("qa_clear_n")
)

qa_opaque_count = (
    qa_diag
    .select("QA60_opaque")
    .sum()
    .rename("qa_opaque_n")
)

qa_cirrus_count = (
    qa_diag
    .select("QA60_cirrus")
    .sum()
    .rename("qa_cirrus_n")
)


diagnostic_stack = (
    qa_valid_count
    .addBands(qa_clear_count)
    .addBands(qa_opaque_count)
    .addBands(qa_cirrus_count)
)


samples = (
    diagnostic_stack
    .sampleRegions(
        collection=year_points,
        properties=[
            "PlotID",
            "Year",
        ],
        scale=60,
        geometries=False,
        tileScale=4,
    )
)


print(
    "Plots returned:",
    samples.size().getInfo()
)


for band in [
    "qa_valid_n",
    "qa_clear_n",
    "qa_opaque_n",
    "qa_cirrus_n",
]:

    print(
        "\n",
        band,
    )

    print(
        samples
        .aggregate_stats(
            band
        )
        .getInfo()
    )

Plots returned: 378

 qa_valid_n
{'max': 68, 'mean': 30.873015873015873, 'min': 9, 'sample_sd': 12.119202900522971, 'sample_var': 146.8750789440444, 'sum': 11670, 'sum_sq': 415660, 'total_count': 378, 'total_sd': 12.103161592498859, 'total_var': 146.48652053413952, 'valid_count': 378, 'weight_sum': 378, 'weighted_sum': 11670}

 qa_clear_n
{'max': 0, 'mean': 0, 'min': 0, 'sample_sd': 0, 'sample_var': 0, 'sum': 0, 'sum_sq': 0, 'total_count': 378, 'total_sd': 0, 'total_var': 0, 'valid_count': 378, 'weight_sum': 378, 'weighted_sum': 0}

 qa_opaque_n
{'max': 38, 'mean': 15.142857142857142, 'min': 5, 'sample_sd': 6.019042662558736, 'sample_var': 36.22887457370215, 'sum': 5724, 'sum_sq': 100336, 'total_count': 378, 'total_sd': 6.011075693266043, 'total_var': 36.133030990173836, 'valid_count': 378, 'weight_sum': 378, 'weighted_sum': 5724}

 qa_cirrus_n
{'max': 40, 'mean': 15.73015873015873, 'min': 3, 'sample_sd': 7.060270317124211, 'sample_var': 49.84741695086521, 'sum': 5946, 'sum_sq': 112324

In [89]:
import rasterio
import numpy as np

tif_path = (
    r"A:\NCA_DATA\S2_2016-2017\gee_ready\S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_20231014T014631_GEE.tif"
)

with rasterio.open(tif_path) as src:

    print("dtype:", src.dtypes)
    print("dataset nodata:", src.nodata)
    print("nodatavals:", src.nodatavals)

    print("\nBand descriptions:")
    for i, desc in enumerate(
        src.descriptions,
        start=1,
    ):
        print(i, desc)

    qa = src.read(12)
    qa_mask = src.read_masks(12)

    print(
        "\nQA60 unique values:",
        np.unique(qa)
    )

    print(
        "QA60 mask unique values:",
        np.unique(qa_mask)
    )

    print(
        "QA60 valid pixels:",
        np.count_nonzero(
            qa_mask > 0
        )
    )

    print(
        "QA60 zero pixels:",
        np.count_nonzero(
            qa == 0
        )
    )

    print(
        "QA60 zero pixels marked valid:",
        np.count_nonzero(
            (qa == 0)
            &
            (qa_mask > 0)
        )
    )

    print(
        "QA60 zero pixels marked invalid:",
        np.count_nonzero(
            (qa == 0)
            &
            (qa_mask == 0)
        )
    )

dtype: ('uint16', 'uint16', 'uint16', 'uint16', 'uint16', 'uint16', 'uint16', 'uint16', 'uint16', 'uint16', 'uint16', 'uint16')
dataset nodata: None
nodatavals: (None, None, None, None, None, None, None, None, None, None, None, None)

Band descriptions:
1 B2
2 B3
3 B4
4 B5
5 B6
6 B7
7 B8
8 B8A
9 B11
10 B12
11 SCL
12 QA60

QA60 unique values: [   0 1024 2048]
QA60 mask unique values: [255]
QA60 valid pixels: 52544964
QA60 zero pixels: 6434064
QA60 zero pixels marked valid: 6434064
QA60 zero pixels marked invalid: 0
